In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# プリミティブの入力と出力

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** 新しい実行モデルのベータ版が公開されました。このDirected実行モデルは、エラー軽減ワークフローをカスタマイズする際の柔軟性を高めます。詳細については、[Directed実行モデル](/guides/directed-execution-model)ガイドをご覧ください。


<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件に基づいて開発されました。
これらのバージョン以降をご使用されることをお勧めします。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

このページでは、IBM Quantum&reg; の計算リソース上でワークロードを実行するQiskit Runtimeプリミティブの入出力の概要を説明します。これらのプリミティブでは、**Primitive Unified Bloc（PUB）** と呼ばれるデータ構造を用いて、ベクトル化されたワークロードを効率的に定義することができます。PUBはQPUがワークロードを実行するための基本的な作業単位です。PUBはSamplerおよびEstimatorプリミティブの[`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run)メソッドへの入力として使用され、定義されたワークロードをジョブとして実行します。ジョブが完了すると、使用したPUBおよびSamplerまたはEstimatorプリミティブで指定したランタイムオプションに応じた形式で結果が返されます。

<span id="pubs"></span>
## PUBの概要
プリミティブの[`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run)メソッドを呼び出す際、必須の引数は1つ以上のタプルからなる`list`で、それぞれのタプルがプリミティブによって実行される各回路に対応します。各タプルはPUBとみなされ、リスト内の各タプルに必要な要素は使用するプリミティブによって異なります。各タプルに与えるデータは、ブロードキャストによりワークロードの柔軟性を高めるため、さまざまな形状に配置することができます。ブロードキャストのルールについては[後述のセクション](#broadcasting-rules)で説明します。

### Estimator PUB
Estimatorプリミティブでは、PUBのフォーマットには最大4つの値を含めることができます。

- 1つの`QuantumCircuit`。1つ以上の[`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)オブジェクトを含めることもできます。
- 1つ以上のオブザーバブルのリスト。推定する期待値を指定し、配列として並べます（例：0次元配列として表現した単一のオブザーバブル、1次元配列として表現したオブザーバブルのリストなど）。データは`Pauli`、`SparsePauliOp`、`PauliList`、`str`などの`ObservablesArrayLike`形式のいずれかで指定できます。
   > **Note:** 同じ回路を持つ異なるPUBに2つの交換可能なオブザーバブルがある場合、それらは同一の測定を使って推定されることはありません。各PUBは異なる測定基底を表すため、PUBごとに個別の測定が必要です。交換可能なオブザーバブルを同じ測定で推定するには、同じPUB内にまとめる必要があります。
- 回路にバインドするパラメータ値のコレクション。最後のインデックスが回路の`Parameter`オブジェクトを指す単一の配列状オブジェクトとして指定するか、回路に`Parameter`オブジェクトがない場合は省略（または等価的に`None`）できます。
- （任意）推定する期待値の目標精度

### Sampler PUB
Samplerプリミティブでは、PUBタプルのフォーマットには最大3つの値を含めることができます。

- 1つの`QuantumCircuit`。1つ以上の[`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)オブジェクトを含めることもできます。
   *注意：これらの回路には、サンプリング対象の各Qubitに対する測定命令も含める必要があります。*
- 回路にバインドするパラメータ値のコレクション $\theta_k$（実行時にバインドする必要がある`Parameter`オブジェクトがある場合のみ必要）
- （任意）回路を測定するショット数

---

以下のコードは、`Estimator`プリミティブへのベクトル化された入力のセット例を示し、それらを単一の`RuntimeJobV2`オブジェクトとしてIBM&reg;バックエンド上で実行するものです。

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV2 as Sampler,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

### ブロードキャストのルール
PUBは複数の配列（オブザーバブルとパラメータ値）の要素をNumPyと同じブロードキャストのルールに従って集約します。このセクションではそれらのルールを簡単にまとめます。詳細については[NumPyブロードキャストルールのドキュメント](https://numpy.org/doc/stable/user/basics.broadcasting.html)をご参照ください。

ルール：

* 入力配列は同じ次元数である必要はありません。
  * 結果の配列は、最も大きい次元数を持つ入力配列と同じ次元数になります。
  * 各次元のサイズは、対応する次元の最大サイズになります。
  * 存在しない次元はサイズ1とみなされます。
* 形状の比較は右端の次元から始まり、左方向に進みます。
* 2つの次元は、それらのサイズが等しいか、一方が1である場合に互換性があります。

ブロードキャスト可能な配列ペアの例：

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

ブロードキャスト不可能な配列ペアの例：

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

`EstimatorV2`はブロードキャストされた形状の各要素に対して1つの期待値の推定値を返します。

以下に、配列のブロードキャストで表現した一般的なパターンの例を示します。それらの視覚的な表現はこの後の図に示されています。

パラメータ値セットは n x m の配列で表され、オブザーバブル配列は1つ以上の単一列配列で表されます。前のコードの各例では、パラメータ値セットとオブザーバブル配列を組み合わせて期待値の推定値を作成しています。

 - *例1*（単一オブザーバブルのブロードキャスト）：パラメータ値セットが5x1の配列で、オブザーバブル配列が1x1です。オブザーバブル配列の1つの要素がパラメータ値セットの各要素と組み合わされ、5x1の配列が生成されます。各要素は、パラメータ値セットの元の要素とオブザーバブル配列の要素の組み合わせです。

 - *例2*（zip）：5x1のパラメータ値セットと5x1のオブザーバブル配列があります。出力は5x1の配列で、各要素はパラメータ値セットのn番目の要素とオブザーバブル配列のn番目の要素の組み合わせです。

  - *例3*（外積/積）：1x6のパラメータ値セットと4x1のオブザーバブル配列があります。これらを組み合わせると4x6の配列が作成され、パラメータ値セットの各要素がオブザーバブル配列の*すべて*の要素と組み合わされます。そのため、各パラメータ値は出力の1列全体になります。

  - *例4*（標準的なnd一般化）：3x6のパラメータ値セット配列と2つの3x1のオブザーバブル配列があります。これらを組み合わせると、前の例と同様の方法で2つの3x6の出力配列が作成されます。

![この画像は配列ブロードキャストのいくつかの視覚的な表現を示しています](../docs/images/guides/primitive-input-output/broadcasting.svg "ブロードキャストの視覚的な表現")

In [4]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=4096, num_bits=10>)

The shape of register `meas` is (4096, 2).

The bytes in register `alpha`, shot by shot:
[[  3 254]
 [  0   0]
 [  3 255]
 ...
 [  0   0]
 [  3 255]
 [  0   0]]



<Admonition type="tip" title="SparsePauliOp">
各`SparsePauliOp`は、`SparsePauliOp`に含まれるPauliの数に関わらず、この文脈では単一の要素として数えられます。したがって、これらのブロードキャストルールの観点では、以下のすべての要素は同じ形状を持ちます。

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111110': 199, '0000000000': 1337, '1111111111': 1052, '1111111000': 33, '1110000000': 65, '1100100000': 2, '1100000000': 25, '0010001110': 1, '0000000011': 30, '1111111011': 58, '1111111010': 25, '0000000110': 7, '0010000001': 11, '0000000001': 179, '1110111110': 6, '1111110000': 33, '1111101111': 49, '1110111111': 40, '0000111010': 2, '0100000000': 35, '0000000010': 51, '0000100000': 31, '0110000000': 7, '0000001111': 22, '1111111100': 24, '1011111110': 5, '0001111111': 58, '0000111111': 24, '1111101110': 10, '0000010001': 5, '0000001001': 2, '0011111111': 38, '0000001000': 11, '1111100000': 34, '0111111111': 45, '0000000100': 18, '0000000101': 2, '1011111111': 11, '1110000001': 13, '1101111000': 1, '0010000000': 52, '0000010000': 17, '0000011111': 15, '1110100001': 1, '0111111110': 9, '0000000111': 19, '1101111111': 15, '1111110111': 17, '0011111110': 5, '0001101110': 1, '0111111011': 6, '0100001000': 2, '0010001111': 1, '1111011000': 1, '0000111110': 4, '0011110010': 1

以下の演算子リストは、含まれる情報という点では同等ですが、異なる形状を持ちます。

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=4096, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=4096, num_bits=9>)


</Admonition>

## プリミティブ出力の概要
1つ以上のPUBがQPUに送信されてジョブが正常に完了すると、データは[`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)コンテナオブジェクトとして返されます。このオブジェクトは`RuntimeJobV2.result()`メソッドを呼び出すことでアクセスできます。`PrimitiveResult`には、各PUBの実行結果を格納した[`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)オブジェクトのイテラブルなリストが含まれます。使用するプリミティブの種類によって、返されるデータはEstimatorの場合は期待値とそのエラーバー、Samplerの場合は回路出力のサンプルとなります。

このリストの各要素は、プリミティブの`run()`メソッドに送信した各PUBに対応しています（たとえば、20個のPUBを含むジョブを送信した場合、20個の`PubResult`を含む`PrimitiveResult`オブジェクトが返され、それぞれが各PUBに対応します）。

各`PubResult`オブジェクトは`data`属性と`metadata`属性を持ちます。`data`属性はカスタマイズされた[`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin)で、実際の測定値や標準偏差などが格納されています。この`DataBin`は、関連するPUBの形状や構造、またジョブ送信に使用したプリミティブが指定するエラー緩和オプション（[ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne)や[PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)など）によってさまざまな属性を持ちます。一方、`metadata`属性にはランタイムやエラー緩和オプションに関する情報が含まれます（詳細はこのページの[結果のメタデータ](#result-metadata)セクションで説明します）。

以下は`PrimitiveResult`データ構造の概略図です：

<Tabs>
    <TabItem value="estimator" label="Estimator Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Sampler Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

簡単に言うと、1つのジョブは`PrimitiveResult`オブジェクトを返し、その中に1つ以上の`PubResult`オブジェクトのリストが含まれます。これらの`PubResult`オブジェクトには、ジョブに送信された各PUBの測定データが格納されています。

各`PubResult`は、使用したプリミティブの種類によって異なる形式と属性を持ちます。詳細は以下で説明します。

### Estimatorの出力
EstimatorプリミティブのPUBに対する各`PubResult`には、少なくとも期待値の配列（`PubResult.data.evs`）と関連する標準偏差（使用した`resilience_level`に応じて`PubResult.data.stds`または`PubResult.data.ensemble_standard_error`）が含まれますが、指定したエラー緩和オプションによってさらに多くのデータが含まれる場合もあります。

以下のコードスニペットは、上記で作成したジョブの`PrimitiveResult`（および関連する`PubResult`）の形式を示しています。

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (4096, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (4096, 2).
The bytes in register `beta`, shot by shot:
[[  0 135]
 [  0 247]
 [  1 247]
 ...
 [  0   0]
 [  1 224]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (4096, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0 135]
 [  0 247]
 [  1 247]
 [  1 128]
 [  1 255]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.068359375
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.06396484375

The shape of the merged results is (4096, 2).
The bytes of the merged results:
[[  1  15]
 [  1 239]
 [  3 239]
 

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'execution' : {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 08:07:33', stop='2026-01-15 08:07:36', size=4096>)])},
'version' : 2,

The metadata of the PubResult result is:
'circuit_metadata' : {},


#### Estimatorによるエラーの計算方法
Estimatorは、入力PUBに含まれるオブザーバブルの平均値の推定（`DataBin`の`evs`フィールド）に加えて、各期待値に関連するエラーの推定値も提供しようとします。すべてのEstimatorクエリは、各期待値に対する標準誤差のような量を`stds`フィールドに格納しますが、`ensemble_standard_error`など追加情報を生成するエラー緩和オプションもあります。

単一のオブザーバブル $\mathcal{O}$ を考えます。[ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne)を使用しない場合、Estimator実行の各ショットは期待値 $\langle \mathcal{O} \rangle$ の点推定を提供すると考えることができます。点ごとの推定値がベクトル`Os`に格納されている場合、`ensemble_standard_error`に返される値は以下と等価です（ここで $\sigma_{\mathcal{O}}$ は[期待値の標準偏差](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2)の推定値、$N_{shots}$ はショット数です）：

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

これはすべてのショットを単一のアンサンブルの一部として扱います。ゲートの[twirling](/guides/error-mitigation-and-suppression-techniques#pauli-twirling)（`twirling.enable_gates = True`）を要求した場合、$\langle \mathcal{O} \rangle$ の点ごとの推定値を共通のtwirlingを共有するセットに分類できます。これらの推定値のセットを`O_twirls`と呼ぶと、`num_randomizations`個（twirlingの数）のセットが得られます。そして`stds`は`O_twirls`の平均値の標準誤差となり、次のように表されます：

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

ここで $\sigma_{\mathcal{O}}$ は`O_twirls`の標準偏差、$N_{twirls}$ はtwirlingの数です。twirlingを有効にしない場合、`stds`と`ensemble_standard_error`は等しくなります。

ZNEを有効にすると、上記の`stds`は外挿モデルへの非線形回帰の重みになります。この場合に最終的に`stds`として返されるのは、ノイズ係数ゼロで評価されたフィットモデルの不確かさです。フィットが不良な場合や不確かさが大きい場合、報告される`stds`は非常に大きくなることがあります。ZNEが有効な場合、`pub_result.data.evs_noise_factors`と`pub_result.data.stds_noise_factors`も設定されるため、独自の外挿処理を行うことができます。

### Samplerの出力
Samplerジョブが正常に完了すると、返される[`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)オブジェクトには、PUBごとに1つの[`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult)のリストが含まれます。これらの`SamplerPubResult`オブジェクトのデータビンは辞書のようなオブジェクトで、回路内の各`ClassicalRegister`に対して1つの`BitArray`が含まれます。

`BitArray`クラスは、順序付きのショットデータのコンテナです。詳しく言うと、サンプリングされたビット列をバイトとして2次元配列に格納します。この配列の左端の軸は順序付きのショットにわたって走り、右端の軸はバイトにわたって走ります。

最初の例として、次の10量子ビット回路を見てみましょう：